# LPI Case Study Group Project Part 1

## Objective: The use of charts to creatively to tell your story is a requirement for your final presentation. Along with representing multiple countries, there must be one time related slide or time series chart in your presentation. That is, you have to track a country (or a group of countries) over time to track the performance in one or more of the six areas to add an interesting plot to your story

##### George's Code:

In [ ]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
# Load the Excel file
file_path = 'International_LPI_from_2007_to_2023_Cleaned.xlsx'

In [ ]:
# Load all sheets into a dictionary of DataFrames

sheets_dict = pd.read_excel(file_path, sheet_name=None)

In [ ]:
# Combine all sheets into one DataFrame
df = pd.concat(sheets_dict.values(), ignore_index=True)

In [ ]:
# Preview the combined DataFrame
print("Combined shape:", df.shape)
df.head()

In [ ]:
# Overview of dataset
df.info()  ## mising 6 country codes

## Exploratory Data Analysis (EDA)

In [ ]:
# Check basic descriptive stats for numeric columns
df.describe().T

In [ ]:
# Check basic descriptive stats for categorical columns
df.describe(include=['object']).T

In [ ]:
# Missing values by column
missing = df.isnull().sum().sort_values(ascending=False)
missing_percent = (missing / len(df)) * 100
missing_df = pd.DataFrame({'Missing Values': missing, '% Missing': missing_percent})
missing_df[missing_df['Missing Values'] > 0]

In [ ]:
# Check for duplicate rows
duplicates = df.duplicated().sum()
print(f" Number of duplicate rows: {duplicates}")

In [ ]:
# Unique values in object columns
for col in df.select_dtypes(include='object').columns:
    print(f"\n{col} - unique values: {df[col].nunique()}")
    print(df[col].unique()[:10])  # show first 10 unique entries

In [ ]:
## Histograms of Numerical Variables

numeric_cols = df.select_dtypes(include=np.number).columns
df[numeric_cols].hist(bins=20, figsize=(15, 10))
plt.suptitle("Distribution of Numeric Variables", fontsize=16)
plt.show()

In [ ]:
# Creat Correlation Matrix

# Select numeric columns
numeric_cols = df.select_dtypes(include=np.number).columns

# Exclude 'Year' and any column containing 'rank'
exclude_cols = [col for col in numeric_cols if ('rank' in col.lower()) or (col.lower() == 'year')]
corr_cols = [col for col in numeric_cols if col not in exclude_cols]

# Compute correlation matrix
corr = df[corr_cols].corr()

# Create interactive correlation matrix
fig = px.imshow(
    corr,
    text_auto=".2f",                # Show correlation values
    color_continuous_scale='RdBu_r', # Red-blue diverging colors
    origin='upper',                  # Align origin to upper-left
    title="Interactive Correlation Matrix (Excluding Year & Rank Columns)"
)

# Improve readability
fig.update_layout(
    width=900,
    height=800,
    xaxis_title="Scores",
    yaxis_title="Scores"
)

fig.show()

In [ ]:
# Detect outliers using z-score
from scipy.stats import zscore

df_numeric = df[numeric_cols].dropna()
z_scores = np.abs(zscore(df_numeric))
outliers = (z_scores > 3).sum()
outliers_df = pd.DataFrame({'Column': numeric_cols, 'Outliers (>3σ)': outliers})
outliers_df

## Explore some visualizations

### Selected a group of large countries across the globe to analyze

In [ ]:
# Focus on a select group of countries
focus_countries = [
    "United States", "Canada", "Chile", "Indonesia",
    "Argentina", "Colombia","Mexico"
]

# Filter DataFrame
focus_df = df[df["Country"].isin(focus_countries)].copy()

print("Filtered dataset shape:", focus_df.shape)
focus_df["Country"].unique()

In [ ]:
# LPI Score Trends

fig = px.line(focus_df, x="Year", y="LPI Score", color="Country",
              title="LPI Score Trend (Selected Countries, 2010–2023)",
              markers=True)
fig.update_yaxes()
fig.show()

In [ ]:
# See which countries improved or decline most 

focus_df = focus_df.sort_values(["Country", "Year"])
df["LPI Change"] = focus_df.groupby("Country")["LPI Score"].diff()

avg_change = focus_df.groupby("Country")["LPI Change"].mean().sort_values(ascending=False)

plt.figure(figsize=(10,6))
sns.barplot(
    x=avg_change.values,
    y=avg_change.index,
    hue=avg_change.values,        # Use hue to map colors
    palette="coolwarm",
    legend=False
)
plt.title("Average Change in LPI Score (2007–2023)")
plt.xlabel("Average Yearly Change")
plt.ylabel("Country")
plt.show()

In [ ]:
# Box and Whisker Plot

plt.figure(figsize=(12,6))
sns.boxplot(data=focus_df, x="Country", y="LPI Score")
plt.title("Distribution of LPI Scores by Country", fontsize=14)
plt.xticks(rotation=45)
plt.show()

In [ ]:

# Filter for Mexico
mexico_df = df[df['Country'].str.contains('Mexico', case=False)].copy()

# Round labels to 2 decimals
mexico_df['LPI Label'] = mexico_df['LPI Score'].round(2).astype(str)

# Plot
fig = px.line(
    mexico_df,
    x='Year',
    y='LPI Score',
    text='LPI Label',
    markers=True,
    title='Mexico- LPI Score Over Time'
)

# Adjust text position and layout
fig.update_traces(
    textposition='top right',   # moves labels slightly above the points
    textfont=dict(size=12
                 )
)

fig.update_layout(
    yaxis_title='LPI Score',
    xaxis_title='Year',
    template='plotly_white',
    height=600
)


fig.show()


In [ ]:
fig = px.line(
    mexico_df,
    x='Year',
    y='LPI Grouped Rank',
    text='LPI Grouped Rank',
    title='Mexico - LPI Global Rank Over Time'
)
fig.update_layout(yaxis_title='Rank (lower is better)', xaxis_title='Year', yaxis_autorange="reversed")
fig.show()

In [ ]:
import plotly.graph_objects as go

# Sort by year to ensure proper sequence
mexico_df = mexico_df.sort_values('Year')

# Calculate year-over-year change
mexico_df['YoY Change'] = mexico_df['LPI Score'].diff()

# Create waterfall trace
fig = go.Figure(go.Waterfall(
    x=mexico_df['Year'],
    y=mexico_df['YoY Change'],
    measure=['relative'] * len(mexico_df),
    text=[f"{v:+.2f}" if pd.notnull(v) else "" for v in mexico_df['YoY Change']],
    textposition='outside',
    connector={"line":{"color":"rgb(63, 63, 63)"}}
))

fig.update_layout(
    title="Mexico - Year-over-Year Change in LPI Score",
    yaxis_title="Change in LPI Score",
    xaxis_title="Year",
    showlegend=False,
    height = 600
)

fig.show()


In [ ]:
# Filter Mexico's data
mexico_df = df[df['Country'].str.contains('Mexico', case=False)]

# Select the relevant component columns
component_cols = [
    'Customs Score', 'Infrastructure Score', 'International Shipments Score',
    'Logistics Competence and Quality Score', 'Timeliness Score', 'Tracking and Tracing Score'
]

# Melt the dataframe for easier plotting
mexico_long = mexico_df.melt(
    id_vars=['Year'],
    value_vars=component_cols,
    var_name='Component',
    value_name='Score'
)

# Plot trends for each component
import plotly.express as px

fig = px.line(
    mexico_long,
    x='Year',
    y='Score',
    color='Component',
    markers=True,
    title="Mexico – LPI Component Scores Over Time"
)
fig.update_layout(yaxis_title="Score", xaxis_title="Year")
fig.show()


In [ ]:
# Density Plot

plt.figure(figsize=(8,4))
sns.kdeplot(df['LPI Score'], fill=True, color='blue')
plt.title('Density Plot of LPI Scores')
plt.xlabel('LPI Score (1-5)')
plt.xlim(1, 5)
plt.show()


#### Ryan's Code:

In [ ]:
# columns present in the dataframe ---
components = [
    "Customs Score",
    "Infrastructure Score",
    "International Shipments Score",
    "Logistics Competence and Quality Score",
    "Timeliness Score",
    "Tracking and Tracing Score",
]

# 1) Filter Mexico and reshape long
mexico_wide = (
    focus_df[focus_df["Country"] == "Mexico"][["Year"] + components]
    .sort_values("Year")
    .dropna(how="all", subset=components)  # safety
)

mexico_long = mexico_wide.melt(
    id_vars="Year", value_vars=components,
    var_name="Component", value_name="Score"
)

# 2) Plot
fig = px.line(
    mexico_long,
    x="Year", y="Score", color="Component",
    markers=True,
    title="Mexico – LPI Component Scores Over Time",
)

# 3) Style to match your look
fig.update_layout(
    template="presentation",
    legend_title_text="Component",
    yaxis_title="Score",
    xaxis_title="Year",
    height=750
)

# Optional: subtle alternating year bands (nice for readability)
years = sorted(mexico_wide["Year"].unique())
for i in range(len(years) - 1):
    if i % 2 == 0:
        fig.add_vrect(
            x0=years[i], x1=years[i+1],
            fillcolor="lightgray", opacity=0.08,
            line_width=0, layer="below"
        )

fig.show()


In [ ]:
# Define country groups
developed_countries = ['United States', 'Canada']
mexico_only = ['Mexico']

# Filter dataset: only keep developed + Mexico
compare_df = focus_df[focus_df['Country'].isin(developed_countries + mexico_only)].copy()

# Create empty list for group labels
groups = []

# Assign group names using if/elif
for country in compare_df['Country']:
    if country in developed_countries:
        groups.append('Developed')
    elif country in mexico_only:
        groups.append('Mexico')

# Add the new list as a column
compare_df['Group'] = groups

# Compute averages for each group by year
group_avg = compare_df.groupby(['Group', 'Year'])['LPI Score'].mean().reset_index()

# Keep only Mexico and Developed rows (just to be extra safe)
group_avg = group_avg[group_avg['Group'].isin(['Mexico', 'Developed'])]

# Plot the chart
fig = px.line(
    group_avg,
    x='Year',
    y='LPI Score',
    height = 750,
    color='Group',
    markers=True,
    template = 'presentation',
    title='Mexico vs Developed Economies — LPI Score Trend (2010–2023)',
    color_discrete_sequence=['#ff7f0e', '#1f77b4']  # orange = Mexico, blue = Developed
)

# Layout tweaks
fig.update_layout(
    yaxis_title='Average LPI Score',
    xaxis_title='Year',
    plot_bgcolor='white',
    title_x=0.5,
    font=dict(size=13)
)

fig.show()

In [ ]:
focus_countries = [
    "Argentina", "Canada",  "Chile", "Colombia",
    "Indonesia", "Mexico", "United States"
]

# Filter DataFrame
focus_df = df[df["Country"].isin(focus_countries)].copy()

print("Filtered dataset shape:", focus_df.shape)
focus_df["Country"].unique()

fig = px.line(focus_df, x="Year", y="LPI Score", color="Country",
              title="LPI Score Trend (Selected Countries, 2010–2023)",
              height = 750,
              template = 'presentation',
              markers=True)
fig.show()

#### Gayle's Code:

In [ ]:
# Focus on a select group of countries
focus_countries = [
    "Namibia", "Botswana", "Rwanda", "Greece", "Somalia"
]

# Filter DataFrame
focus_df = df[df["Country"].isin(focus_countries)].copy()

print("Filtered dataset shape:", focus_df.shape)
focus_df["Country"].unique()

In [ ]:
#Interactive Scatterplot

x_col = "Customs Score"
y_col = "Infrastructure Score"

fig = px.scatter(
    focus_df,
    x=x_col,
    y=y_col,
    color="Country",
    hover_data={
        "Country": True,
        "Year": True,
        x_col: ":.2f",
        y_col: ":.2f",
        "LPI Score": ":.2f"
    },
    title="Infrastructure vs Customs for Focus Countries",
    size="LPI Score",
    size_max=15,
)

fig = px.scatter(
    focus_df,
    x=x_col,
    y=y_col,
    color="Country",
    hover_data=["Country", "Year", x_col, y_col, "LPI Score"],
    trendline="ols", 
    title="Infrastructure vs Customs for Focus Countries",
)

fig.update_layout(
    width=900,
    height=500,
    title_font_size=16,
    plot_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="lightgray"),
    yaxis=dict(showgrid=True, gridcolor="lightgray"),
)

fig.show()

In [ ]:
# Geographical choropleth map
year = 2023
map_df = df[df["Year"] == year]

fig = px.choropleth(
    map_df,
    locations="Code",
    color="LPI Score",
    hover_name="Country",
    color_continuous_scale="Viridis",
    projection="equirectangular",
    title=f"LPI Score by Country — {year}"
)
fig.update_layout(title_font_size=16, margin=dict(l=0, r=0, t=60, b=0))
fig.show()

#### Ivy's Code:

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


file_path = "International_LPI_from_2007_to_2023_Cleaned.xlsx"
years = [2010, 2012, 2014, 2016, 2018, 2023]
developed = ["DEU", "JPN", "USA", "SWE", "SGP"]              # developed countries
developing = ["CHL", "MEX", "IDN", "ARG", "COL"]             # developing countries(and ex.MEX for nearshoring)

# Read every sheet(years)
xls = pd.ExcelFile(file_path)
frames = []
for s in xls.sheet_names:
    df = pd.read_excel(file_path, sheet_name=s)
    df.columns = [c.strip().replace(" ", "_").lower() for c in df.columns]
    df = df.rename(columns={
        "country_code": "code",
        "countrycode": "code",
        "lpi": "lpi_score",
        "lpi_score_": "lpi_score"
    })
    frames.append(df)

data = pd.concat(frames, ignore_index=True)

# checking
for col in ["code","year","lpi_score","country"]:
    if col not in data.columns:
        raise ValueError(f"Missing expected column: {col}. Actual columns: {list(data.columns)}")


data = data[data["year"].isin(years) & data["code"].isin(developed + developing)].copy()
data["group"] = data["code"].apply(lambda x: "Developed" if x in developed else "Developing")


avg = (data.groupby(["group","year"])["lpi_score"].mean().reset_index())

plt.figure(figsize=(8,5))
for g, sub in avg.groupby("group"):
    plt.plot(sub["year"], sub["lpi_score"], marker="o", linewidth=3,
             label=g, color=("steelblue" if g=="Developed" else "orange"))
plt.ylim(2.8, 4.2)  
plt.title("Developed vs Developing — Average LPI Score (2010–2023)")
plt.xlabel("Year"); plt.ylabel("Average LPI Score")
plt.grid(alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# === 檔案設定 ===
file_path = "International_LPI_from_2007_to_2023_Cleaned.xlsx"

developing = ["CHL", "MEX", "IDN", "ARG", "COL"]
developed = ["DEU", "JPN", "USA", "SWE", "SGP"]

# === 讀檔 ===
xls = pd.ExcelFile(file_path)
frames = []
for s in xls.sheet_names:
    df = pd.read_excel(file_path, sheet_name=s)
    df.columns = [c.strip().replace(" ", "_").lower() for c in df.columns]
    df = df.rename(columns={"country_code": "code"})
    frames.append(df)

lpi = pd.concat(frames, ignore_index=True)
lpi["year"] = pd.to_numeric(lpi["year"], errors="coerce")

# === 篩選特定國家與年份 ===
target_countries = developing + developed
data = lpi[(lpi["year"].isin([2010, 2012, 2014, 2016, 2018, 2023])) & (lpi["code"].isin(target_countries))]

# === 標記群組 ===
data["Group"] = data["code"].apply(lambda x: "Developed" if x in developed else "Developing")

# === 計算每組平均 LPI ===
avg_trend = data.groupby(["Group", "year"])["lpi_score"].mean().reset_index()

# === 繪製折線圖 ===
plt.figure(figsize=(8,5))
for group, df in avg_trend.groupby("Group"):
    plt.plot(df["year"], df["lpi_score"], marker="o", linewidth=2,
             label=group, color="steelblue" if group=="Developed" else "orange")

plt.title("Developed vs Developing Economies — Average LPI Score Trend (2010–2023)")
plt.xlabel("Year")
plt.ylabel("Average LPI Score")
plt.grid(True, linestyle="--", alpha=0.6)
plt.legend(title="Group")
plt.tight_layout()
plt.show()

In [ ]:
asia_iso3 = [
    # East Asia
    "CHN","HKG","JPN","KOR","MAC","MNG","TWN","SGP",
    # Southeast Asia
    "IDN","MYS","PHL","THA","VNM","KHM","LAO","MMR","BRN","TLS",
    # South Asia
    "IND","PAK","BGD","LKA","NPL","BTN","MDV"
]
asia = lpi[lpi["code"].isin(asia_iso3)].copy()
asia["country"] = asia["country"].str.strip()
asia.sample(5)

In [ ]:
# 先算每個國家每年的 LPI（已經是單值，但保險用 mean）
ts = (asia.groupby(["country","year"])["lpi_score"]
           .mean()
           .reset_index()
           .sort_values(["country","year"]))

# 第一年與最後一年分數
summary = (ts.groupby("country")
             .agg(first_year=("year","min"),
                  last_year=("year","max"),
                  start=("lpi_score","first"),
                  end=("lpi_score","last"))
             .reset_index())
summary["change"] = summary["end"] - summary["start"]
summary["pct_change_%"] = summary["change"]/summary["start"]*100

top_inc = summary.sort_values("change", ascending=False).head(8)
top_dec = summary.sort_values("change", ascending=True).head(8)

top_inc, top_dec

In [ ]:
def plot_lines(df, countries, title, outfile):
    pivot = (df[df["country"].isin(countries)]
             .pivot_table(index="year", columns="country", values="lpi_score", aggfunc="mean")
            )
    plt.figure()
    pivot.plot(marker="o")
    plt.title(title)
    plt.xlabel("Year")
    plt.ylabel("LPI Score")
    plt.legend(title="Country", bbox_to_anchor=(1.05,1), loc="upper left")
    plt.tight_layout()
    plt.show()

plot_lines(
    ts, top_inc["country"].tolist(),
    "Asia — Top Improvers in LPI (first → last available year)",
    "asia_top_improvers_timeseries.png"
)

plot_lines(
    ts, top_dec["country"].tolist(),
    "Asia — Largest Declines in LPI (first → last available year)",
    "asia_largest_declines_timeseries.png"
)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# === 檔案設定 ===
file_path = "International_LPI_from_2007_to_2023_Cleaned.xlsx"

developing = ["CHL", "MEX", "IDN", "ARG", "COL"]
developed = ["DEU", "JPN", "USA", "SWE", "SGP"]

# === 讀檔 ===
xls = pd.ExcelFile(file_path)
frames = []
for s in xls.sheet_names:
    df = pd.read_excel(file_path, sheet_name=s)
    df.columns = [c.strip().replace(" ", "_").lower() for c in df.columns]
    df = df.rename(columns={"country_code": "code"})
    frames.append(df)

lpi = pd.concat(frames, ignore_index=True)
lpi["year"] = pd.to_numeric(lpi["year"], errors="coerce")

# === 篩出目標國家 & 年份 ===
target_countries = developing + developed
data = lpi[(lpi["year"].isin([2018, 2023])) & (lpi["code"].isin(target_countries))]

# === 標記群組 ===
data["Group"] = data["code"].apply(lambda x: "Developed" if x in developed else "Developing")

# === 整理成比較表 ===
pivot = data.pivot_table(index=["country", "Group"], columns="year", values="lpi_score").reset_index()
pivot["Change"] = pivot[2023] - pivot[2018]
pivot = pivot.sort_values("Change")

# === 顏色設定 ===
colors = pivot["Group"].map({"Developed":"steelblue","Developing":"orange"})

# === 視覺化 ===
plt.figure(figsize=(8,6))
plt.barh(pivot["country"], pivot["Change"], color=colors)
plt.axvline(0, color="black", linewidth=1)
plt.title("LPI Score Change (2018 → 2023): Developed vs Developing Economies")
plt.xlabel("Change in LPI Score")
plt.ylabel("Country")
plt.tight_layout()
plt.show()

pivot

#### Mari's Code:

In [ ]:
## Yearly Avg Changes

sheets_dict = pd.read_excel(file_path, sheet_name=None)
df = pd.concat(sheets_dict.values(), ignore_index=True)

focus_countries = [
    "Argentina", "Canada",  "Chile", "Colombia",
    "Indonesia", "Mexico", "United States"
]

# Filter DataFrame
focus_df = df[df["Country"].isin(focus_countries)].copy()

g = (focus_df.sort_values(["Country", "Year"])
     .groupby("Country")["LPI Score"]
     .agg(first="first", last = "last"))
g["years"] = focus_df.groupby("Country")["Year"].agg(lambda x: x.max() - x.min())
g["avg_yearly_change"] = (g["last"] - g["first"]) / g["years"].replace(0,1)

trend_summary = g[["avg_yearly_change"]].sort_values("avg_yearly_change", ascending=False)
display(trend_summary.head(5))

In [ ]:
## Covid Chart

fig_simple = px.line(
    focus_df, x="Year", y="LPI Score", color="Country",
    markers=True, title="LPI Over Time (Clean View)"
)
fig_simple.update_layout(template="plotly_white", hovermode="x unified")

# I saw on yalls some interesting changes during covid times so i added this to zoom in on that
fig_simple.add_vrect(x0=2019, x1=2021, fillcolor="lightgray", opacity=0.25,
                     annotation_text="COVID", annotation_position="top left")

fig_simple.show()